# Lab 18: LLM as a Judge

When we have a robust evaluation dataset (> 100 inputs) then eventually ti will start getting difficult to manually review hundreds of evaluation outputs.

In this lab, we will see how we can use an LLM to evaluate the outputs. This method of evaluation is called *LLM as a Judge*.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model="gpt-4o")

As before, we have this prompt that we want to improve

In [ ]:
from langchain_core.prompts import PromptTemplate

IMPROVE_JD_PROMPT = """
You are a human resource specialist. Your goal is to improve the given job description.

## Instructions

- Always mention the salary, location and job title
  - If salary is not mentioned, then specify "Competitive salary as per local market rates"

<job_description>
{jd}
</job_description>

Answer:
"""

chain = PromptTemplate.from_template(IMPROVE_JD_PROMPT) | llm

And we are going to test the prompt with this job description

In [ ]:
jd = """
We’re seeking a Forward Deployed Engineer. 

We want someone with 3+ years of software engineering experience with production systems. 

They should be rockstar programmers and problem solvers fluent with AI SDLC, Agentic Harnessess and Agent Engineering. 

Ability to think out of the box and exceptional prior experience is valued.

They should have experience in a customer-facing technical role with a background in systems integration or professional services
"""

response = chain.invoke({"jd": jd})

print(response.content)

## LLM as a Judge

Instead of evaluating the above output manually, we are going to create a Judge to evaluate it.

Below is a prompt that we can give to the judge

In [ ]:
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel
from typing import Literal

JD_STRUCTURE_JUDGE_PROMPT = """
You are a job description reviewer. You goal is to review the structure of the given job description and return whether it follows the correct structure or not.

The correct structure follows this format:

1. Job overview is mentioned in the first sentence
1. This is followed by the job responsibilities
1. Then we have the must have skills followed by nice to have skills
1. Finally the benefits of what the company offers are mentioned

After evaluating the job description structure, choose one of the following outputs:

- Bad: Less than three parts of the structure are followed
- Average: Three parts of the structure are followed
- Excellent: All four parts of the structure are followed

Along with the evaluation, return a reason for the choice

This is the job description to evaluate

{revised_job_description}
"""

prompt = PromptTemplate.from_template(JD_STRUCTURE_JUDGE_PROMPT)

We can now ask the LLM to evaluate the output

In [ ]:
class StructureJudgeResponse(BaseModel):
    evaluation: Literal["Bad", "Average", "Excellent"]
    reason: str

chain = prompt | llm.with_structured_output(StructureJudgeResponse)

evaluation_response = chain.invoke({"revised_job_description": response.content})

In [ ]:
print(evaluation_response.evaluation)

In [ ]:
print(evaluation_response.reason)